# Aim 1 replication — Kang 2018 analysis (corrected pipeline)

**Data:** GSE96583 batch 2 (downloaded with `aim1_replication_kang_randolph.ipynb` into `data/kang2018/`). 8 lupus patients, PBMCs pooled and cultured together, control vs IFN-beta for 6 h.

**Finding to test (Parse):** in lymphocytes, typical donors differ little in response *shape*; pairs involving a high-baseline-interferon donor differ much more in shape.

**Pipeline:** singlets only; 6 cell types; random halves of cells; **equal cell counts** per cell type (binomial thinning); **cross-half estimator**. Because all donors share wells and processing, no date or well correction is needed.

**Tests:** Spearman(baseline IFN score, shape deviation) with an exact permutation p (all 8! orderings); top-3 IFN group vs others with an exact permutation p (56 groups, smallest p = 1/56).

**Caveat:** the stimulus is IFN-beta itself, so high-baseline donors may respond less (saturation); read size and shape together.

In [1]:
import os
import numpy as np, pandas as pd

KANG_DIR = "data/kang2018"
RESULTS_DIR = "results/aim1_replication2"
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. Load and pseudobulk

In [2]:
# ===== 1. Kang 2018: load and pseudobulk =====
# Barcodes repeat between the control and stimulated libraries, so cells are matched on (barcode, condition).
import scipy.sparse as sp
from scipy.io import mmread
from scipy.stats import spearmanr
from itertools import combinations, permutations

CELL_TYPES_K = ["CD4 T cells", "CD8 T cells", "B cells", "NK cells", "CD14+ Monocytes", "FCGR3A+ Monocytes"]
SEED = 0

genes_k = pd.read_csv(os.path.join(KANG_DIR, "GSE96583_batch2.genes.tsv.gz"), sep="\t", header=None)
symbols_k = genes_k.iloc[:, 1 if genes_k.shape[1] > 1 else 0].astype(str).to_numpy()
km = pd.read_csv(os.path.join(KANG_DIR, "GSE96583_batch2.total.tsne.df.tsv.gz"), sep="\t")
if "barcode" not in km.columns:                       # barcodes are the row index in the GEO file
    km = km.rename_axis("barcode").reset_index()
km = km[(km.multiplets == "singlet") & km.cell.isin(CELL_TYPES_K)]
km["donor"] = km["ind"].astype(str)

rng_h = np.random.default_rng(SEED)
parts, meta_rows = [], []
for gsm, lib, cond in [("GSM2560248", "2.1", "ctrl"), ("GSM2560249", "2.2", "stim")]:
    X = mmread(os.path.join(KANG_DIR, f"{gsm}_{lib}.mtx.gz"))
    X = (X.T if X.shape[0] == len(symbols_k) else X).tocsr()            # cells x genes
    bcs = pd.read_csv(os.path.join(KANG_DIR, f"{gsm}_barcodes.tsv.gz"), header=None)[0].astype(str)
    m = km[km.stim == cond].set_index("barcode").reindex(bcs)
    keep = m.donor.notna().to_numpy()
    half = rng_h.integers(0, 2, len(bcs))
    key = pd.Series(list(zip(m.donor, m.cell, [cond] * len(bcs), half))).where(keep)
    code_, uniq = pd.factorize(key)
    G = sp.csr_matrix((np.ones(keep.sum()), (code_[keep], np.where(keep)[0])), shape=(len(uniq), len(bcs)))
    parts.append((G @ X).toarray())
    meta_rows += [dict(donor=u[0], cell_type=u[1], condition=u[2], half=u[3], n_cells=int(n))
                  for u, n in zip(uniq, np.bincount(code_[keep], minlength=len(uniq)))]
KSUMS = np.vstack(parts)
KMETA = pd.DataFrame(meta_rows)
print(KMETA.pivot_table(index=["donor"], columns=["condition"], values="n_cells", aggfunc="sum"))

condition  ctrl  stim
donor                
101         826  1085
1015       2688  2247
1016       1696  1553
1039        421   572
107         503   494
1244       1818  1393
1256       2068  1937
1488       1970  2408


## 2. Baseline IFN score and size/shape deviations

In [3]:
# ===== 2. Kang 2018: baseline IFN score and corrected size/shape deviations =====
# All donors were pooled and stimulated together, so well noise and processing are shared by every donor and
# cancel in donor-to-donor comparisons; no date or well correction is needed.
ISG = ["IFI27", "IFI44L", "IFI44", "IFI6", "IFIT1", "IFIT2", "IFIT3", "ISG15", "MX1", "MX2",
       "OAS1", "OAS2", "OAS3", "RSAD2", "HERC5", "XAF1", "EPSTI1", "CMPK2", "STAT1", "IFITM3"]
MIN_EQUAL_HALF = 15; MIN_CPM = 20; N_GENES = 2000
rng_thin = np.random.default_rng(SEED)

def logcpm(x): return np.log1p(x / max(x.sum(), 1) * 1e6)
def prof(d, ct, cond, h=None):
    m = (KMETA.donor == d) & (KMETA.cell_type == ct) & (KMETA.condition == cond)
    if h is not None: m &= KMETA.half == h
    r = KMETA[m]; return KSUMS[r.index.to_numpy()].sum(0), int(r.n_cells.sum())
def thin(c, n_from, n_to):
    if n_from <= n_to: return c.astype(np.float64)
    return rng_thin.binomial(np.round(c).astype(np.int64), n_to / n_from).astype(np.float64)
def cross_parts(Ad, Bd, Ae, Be):
    nd, ne = np.sqrt(max(Ad @ Bd, 1e-12)), np.sqrt(max(Ae @ Be, 1e-12))
    cos = np.clip((Ad @ Be + Bd @ Ae) / 2 / (nd * ne), -1, 1)
    return (nd - ne) ** 2, 2 * nd * ne * (1 - cos)

donors_k = sorted(KMETA.donor.unique())
# baseline IFN score: mean z-scored ISG expression in control cells, averaged over cell types
gi_isg = [i for i, g in enumerate(symbols_k) if g in set(ISG)]
per_ct = {}
for ct in CELL_TYPES_K:
    M = pd.DataFrame({d: logcpm(prof(d, ct, "ctrl")[0])[gi_isg] for d in donors_k if prof(d, ct, "ctrl")[1] >= 20}).T
    per_ct[ct] = ((M - M.mean(axis=0)) / (M.std(axis=0) + 1e-9)).mean(axis=1)
KANG_IFN = pd.DataFrame(per_ct).mean(axis=1).rename("ifn_score").sort_values(ascending=False)
N_HIGH = 3                                             # ~ the Parse fraction (4 of 12)
IFN_HIGH_K = list(KANG_IFN.index[:N_HIGH])
KANG_IFN.to_csv(f"{RESULTS_DIR}/kang_baseline_ifn_score.csv")
print("baseline IFN score:\n", KANG_IFN.round(2), "\nhigh-IFN donors:", IFN_HIGH_K)

pairs, cells_used = [], []
for ct in CELL_TYPES_K:
    raw = {}
    for d in donors_k:
        p = {k: prof(d, ct, c, h) for k, c, h in [("sA", "stim", 0), ("sB", "stim", 1), ("uA", "ctrl", 0), ("uB", "ctrl", 1)]}
        if min(n for _, n in p.values()) >= MIN_EQUAL_HALF: raw[d] = p
    if len(raw) < 6: continue
    t = min(n for p in raw.values() for _, n in p.values())
    R = {}
    for d, p in raw.items():
        th = {k: thin(s, n, t) for k, (s, n) in p.items()}
        R[d] = (logcpm(th["sA"]) - logcpm(th["uA"]), logcpm(th["sB"]) - logcpm(th["uB"]))
    ut = np.mean([logcpm(prof(d, ct, "ctrl")[0]) for d in raw], 0)
    expressed = np.expm1(ut) >= MIN_CPM
    F = np.array([(a + b) / 2 for a, b in R.values()])
    gidx = np.argsort(np.where(expressed, F.var(0), -np.inf))[::-1][:min(N_GENES, expressed.sum())]
    A = {d: R[d][0][gidx] for d in R}; B = {d: R[d][1][gidx] for d in R}
    energy = np.mean([A[d] @ B[d] for d in R])
    cells_used.append(dict(cell_type=ct, donors=len(R), cells_per_half=t))
    for d, e in combinations(sorted(R), 2):
        m, s = cross_parts(A[d], B[d], A[e], B[e])
        k = int(d in IFN_HIGH_K) + int(e in IFN_HIGH_K)
        pairs.append(dict(cell_type=ct, d=d, e=e, size=m / energy, shape=s / energy,
                          pair=["non x non", "high x non", "high x high"][k]))
KANG_PAIRS = pd.DataFrame(pairs)
KANG_PAIRS.to_csv(f"{RESULTS_DIR}/kang_pairs.csv", index=False)
print(pd.DataFrame(cells_used).set_index("cell_type"))

baseline IFN score:
 1015    0.86
1016    0.64
1256    0.58
1039    0.29
101    -0.44
1488   -0.45
1244   -0.47
107    -0.96
Name: ifn_score, dtype: float64 
high-IFN donors: ['1015', '1016', '1256']
                   donors  cells_per_half
cell_type                                
CD4 T cells             8              73
CD8 T cells             6              15
B cells                 6              39
NK cells                7              19
CD14+ Monocytes         8              58
FCGR3A+ Monocytes       7              16


## 3. Tests

In [4]:
# ===== 3. Kang 2018: tests =====
def pair_summary(g):
    s = g[["size", "shape"]].mean(); tot = max(s["size"], 0) + max(s["shape"], 0)
    return pd.Series({"size": s["size"], "shape": s["shape"], "share_size": max(s["size"], 0) / tot if tot > 0 else np.nan})
KANG_BY_PAIR = KANG_PAIRS.groupby(["cell_type", "pair"]).apply(pair_summary).round(3)
KANG_BY_PAIR.to_csv(f"{RESULTS_DIR}/kang_by_pair.csv")
display(KANG_BY_PAIR)

rows = []
for ct, g in KANG_PAIRS.groupby("cell_type"):
    long = pd.concat([g[["d", "shape", "size"]].rename(columns={"d": "donor"}), g[["e", "shape", "size"]].rename(columns={"e": "donor"})])
    dev = long.groupby("donor")[["shape", "size"]].mean().join(KANG_IFN)
    rho_shape = spearmanr(dev.ifn_score, dev["shape"])[0]; rho_size = spearmanr(dev.ifn_score, dev["size"])[0]
    # exact permutation p for the correlation (all orderings of the scores) and for the top-3 group contrast
    sc, sh = dev.ifn_score.to_numpy(), dev["shape"].to_numpy()
    null_rho = [spearmanr(np.array(p), sh)[0] for p in permutations(sc)] if len(sc) <= 8 else []
    high = [d for d in dev.index if d in IFN_HIGH_K]
    contrast = dev.loc[high, "shape"].mean() - dev.drop(high)["shape"].mean()
    null_c = [dev.loc[list(s), "shape"].mean() - dev.drop(list(s))["shape"].mean() for s in combinations(dev.index, len(high))]
    rows.append(dict(cell_type=ct, n_donors=len(dev), rho_shape=rho_shape, rho_size=rho_size,
                     p_rho_shape=np.mean(np.array(null_rho) >= rho_shape) if null_rho else np.nan,
                     high_minus_other_shape=contrast, p_contrast=np.mean(np.array(null_c) >= contrast)))
KANG_TEST = pd.DataFrame(rows).set_index("cell_type").round(3)
KANG_TEST.to_csv(f"{RESULTS_DIR}/kang_tests.csv")
KANG_TEST

size  shape  share_size
cell_type         pair                                 
B cells           high x high  0.004  0.877       0.005
                  high x non   0.011  0.729       0.015
                  non x non    0.028  0.561       0.047
CD14+ Monocytes   high x high  0.009  0.037       0.202
                  high x non   0.023  0.063       0.268
                  non x non    0.007  0.098       0.063
CD4 T cells       high x high  0.016  0.335       0.046
                  high x non   0.008  0.421       0.020
                  non x non    0.005  0.365       0.014
CD8 T cells       high x high  0.058  1.933       0.029
                  high x non   0.040  1.986       0.020
                  non x non    0.057  1.754       0.031
FCGR3A+ Monocytes high x high  0.011  0.215       0.048
                  high x non   0.033  0.440       0.069
                  non x non    0.016  0.477       0.032
NK cells          high x high  0.040  1.554       0.025
                  high x non   0.027  1.619       0.016
                  non x non    0.036  1.499       0.023

,n_donors,rho_shape,rho_size,p_rho_shape,high_minus_other_shape,p_contrast
cell_type,,,,,,
B cells,6,0.486,-0.771,0.178,0.126,0.100
CD14+ Monocytes,8,-0.405,0.024,0.850,-0.028,0.929
CD4 T cells,8,-0.333,0.524,0.805,0.007,0.446
CD8 T cells,6,0.200,-0.086,0.357,0.072,0.450
FCGR3A+ Monocytes,7,-0.679,-0.393,0.956,-0.094,0.914
NK cells,7,0.107,0.179,0.420,0.038,0.400
